<a href="https://colab.research.google.com/github/Viswas-Vinayakumar/AI-search/blob/main/notebooks/Semantic_search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

** SEMANTIC SEARCH **

In [ ]:
!pip -q install sentence-transformers
import numpy as np
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

# Steps 1-2: one vector PER TOKEN
tok_vecs = model.encode("Gehalt verhandeln", output_value="token_embeddings")
print("token vectors:", tok_vecs.shape)              # (number of tokens, 384)

# Step 3 by hand: mean pooling = average over the tokens
mine = tok_vecs.mean(axis=0).cpu().numpy()
lib  = model.encode("Gehalt verhandeln")              # the library's own result
print("same as library?", np.allclose(mine, lib, atol=1e-4))

# Steps 4 + 7 by hand: normalise, then dot product = cosine
def cos(a, b):
    a, b = a / np.linalg.norm(a), b / np.linalg.norm(b)
    return float(a @ b)

q = model.encode("Gehalt verhandeln")
for text in ["Berufliche Neuorientierung, Bewerbungsstrategie und Gehaltsverhandlung für Fachkräfte",
             "Das Harvard-Konzept: Preisverhandlungen im B2B-Vertrieb",
             "Excel für Controlling: Pivot-Tabellen, SVERWEIS"]:
    print(f"{cos(q, model.encode(text)):.3f}  {text[:50]}")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

token vectors: torch.Size([7, 384])
same as library? True
0.321  Berufliche Neuorientierung, Bewerbungsstrategie un
0.358  Das Harvard-Konzept: Preisverhandlungen im B2B-Ver
0.004  Excel für Controlling: Pivot-Tabellen, SVERWEIS


Tokens

In [ ]:
print(model.tokenizer.convert_ids_to_tokens(model.tokenizer("Gehalt verhandeln")["input_ids"]))

['<s>', '▁Ge', 'halt', '▁ver', 'handel', 'n', '</s>']


Reason for wrong prediction :

The salary text covers three topics: career change, application strategy and salary negotiation. Its final vector is the average of all of them, so only about a third of it is “salary negotiation”. The Harvard text is almost 100% about negotiation, so its vector points strongly in that one direction.

Gehalt verhandeln” is two ideas. The model matches the “verhandeln” half very strongly to the pure negotiation text.

MiniLM is fast and light, but not the most precise.

Chunking :

In [ ]:
parts = ["Berufliche Neuorientierung",
         "Bewerbungsstrategie",
         "Gehaltsverhandlung für Fachkräfte"]
scores = [cos(q, model.encode(p)) for p in parts]
for s, p in zip(scores, parts):
    print(f"{s:.3f}  {p}")
print("best part:", round(max(scores), 3), " vs  Harvard text: 0.358")

0.394  Berufliche Neuorientierung
0.336  Bewerbungsstrategie
0.482  Gehaltsverhandlung für Fachkräfte
best part: 0.482  vs  Harvard text: 0.358


Chunking worked. On its own, the salary part scores 0.482 and clearly beats the Harvard text. The dilution was the cause. Averaging three topics hid the one that mattered.

Other fixes, best first:

Chunking (above)

A cross-encoder reranker that reads query and profile together, like your NLI model

A stronger multilingual embedding model

German compound splitting for BM25, so keyword search catches “Gehaltsverhandlung” too